# Avaliação histórica dos sinais de aumento incomum

## Objetivo

Avaliar retrospectivamente o comportamento da regra V1 de sinais de
aumento incomum, respeitando a ordem temporal dos dados.

A avaliação busca responder:

- quantos sinais a regra teria produzido no passado;
- em quais períodos esses sinais ocorreriam;
- quantos coincidem com eventos de referência definidos
  independentemente da regra;
- quantos sinais ocorreriam fora desses eventos;
- com que antecedência um sinal apareceria em relação ao início de um
  evento de referência.

A avaliação deve evitar vazamento de informação futura.

Um `SIGNAL` representa somente o atendimento aos critérios estatísticos
da regra V1. Ele não constitui confirmação de surto ou evento
epidemiológico.

## Separação temporal

Para evitar avaliar a regra nos mesmos períodos utilizados para sua
inspeção inicial, o histórico será dividido temporalmente em:

- desenvolvimento: 2019–2023;
- avaliação: 2024–2025;
- série corrente: 2026, não utilizada neste backtest.

A separação é temporal, e não aleatória, para preservar a ordem natural
da série e reduzir vazamento de informações futuras.

In [11]:
from pathlib import Path

import pandas as pd

In [13]:
project_root = Path.cwd().parent

assert (project_root / "data").exists(), (
    f"Diretório data não encontrado em {project_root}"
)

print(f"Raiz do projeto: {project_root}")

Raiz do projeto: /home/matheuspessoa/Sentinela


In [19]:
historical_weekly_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_weekly_2019_2026.csv"
)

assert historical_weekly_path.exists(), (
    f"Arquivo não encontrado: {historical_weekly_path}"
)

historical_weekly = pd.read_csv(
    historical_weekly_path,
    sep=";",
)

required_columns = {
    "epi_year",
    "epi_week",
    "record_count",
}

assert required_columns.issubset(
    historical_weekly.columns
), "Colunas obrigatórias ausentes."

assert not historical_weekly.duplicated(
    ["epi_year", "epi_week"]
).any()

print(f"Arquivo: {historical_weekly_path.name}")

print(
    "Anos disponíveis:",
    sorted(historical_weekly["epi_year"].unique()),
)

print(
    "Linhas semanais:",
    len(historical_weekly),
)

print("✓ Série histórica semanal carregada.")

Arquivo: srag_mg_historical_weekly_2019_2026.csv
Anos disponíveis: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Linhas semanais: 404
✓ Série histórica semanal carregada.


In [20]:
DEVELOPMENT_YEARS = list(range(2019, 2024))
EVALUATION_YEARS = [2024, 2025]

MIN_HISTORICAL_YEARS = 3
SIGNAL_FACTOR = 1.5

print("Desenvolvimento:", DEVELOPMENT_YEARS)
print("Avaliação:", EVALUATION_YEARS)

Desenvolvimento: [2019, 2020, 2021, 2022, 2023]
Avaliação: [2024, 2025]


In [23]:
def build_reference_before_year(
    weekly_data: pd.DataFrame,
    target_year: int,
) -> pd.DataFrame:

    past = weekly_data.loc[
        (weekly_data["epi_year"] < target_year)
        & (weekly_data["epi_week"].between(1, 52))
    ].copy()

    reference = (
        past
        .groupby("epi_week")["record_count"]
        .agg(
            historical_median="median",
            q75=lambda x: x.quantile(0.75),
            historical_years="count",
        )
        .reset_index()
    )

    reference["has_minimum_history"] = (
        reference["historical_years"]
        >= MIN_HISTORICAL_YEARS
    )

    return reference

In [24]:
reference_2024 = build_reference_before_year(
    historical_weekly,
    2024,
)

reference_2025 = build_reference_before_year(
    historical_weekly,
    2025,
)

assert reference_2024["historical_years"].eq(5).all()
assert reference_2025["historical_years"].eq(6).all()

print("✓ Referências temporais construídas sem informação futura.")
print(
    "2024:",
    reference_2024["historical_years"].min(),
    "anos anteriores",
)
print(
    "2025:",
    reference_2025["historical_years"].min(),
    "anos anteriores",
)

✓ Referências temporais construídas sem informação futura.
2024: 5 anos anteriores
2025: 6 anos anteriores


In [28]:
def evaluate_year(
    weekly_data: pd.DataFrame,
    target_year: int,
) -> pd.DataFrame:

    current = (
        weekly_data.loc[
            (
                (weekly_data["epi_year"] == target_year)
                & (weekly_data["epi_week"].between(1, 52))
            ),
            [
                "epi_year",
                "epi_week",
                "record_count",
            ],
        ]
        .copy()
    )

    reference = build_reference_before_year(
        weekly_data,
        target_year,
    )

    evaluation = current.merge(
        reference,
        on="epi_week",
        how="left",
        validate="one_to_one",
    )

    evaluation["median_threshold"] = (
        evaluation["historical_median"]
        * SIGNAL_FACTOR
    )

    evaluation["ratio_to_median"] = (
        evaluation["record_count"]
        / evaluation["historical_median"]
    )

    insufficient_data = (
        evaluation["historical_median"].isna()
        | evaluation["q75"].isna()
        | ~evaluation["has_minimum_history"].fillna(False)
    )

    signal = (
        ~insufficient_data
        & (evaluation["record_count"] > evaluation["q75"])
        & (
            evaluation["record_count"]
            >= evaluation["median_threshold"]
        )
    )

    evaluation["signal_status"] = "NO_SIGNAL"

    evaluation.loc[
        insufficient_data,
        "signal_status",
    ] = "INSUFFICIENT_DATA"

    evaluation.loc[
        signal,
        "signal_status",
    ] = "SIGNAL"

    return evaluation

In [29]:
backtest_frames = [
    evaluate_year(
        historical_weekly,
        year,
    )
    for year in EVALUATION_YEARS
]

backtest = (
    pd.concat(
        backtest_frames,
        ignore_index=True,
    )
    .sort_values(
        ["epi_year", "epi_week"]
    )
    .reset_index(drop=True)
)

assert set(backtest["epi_year"]) == {2024, 2025}

assert not backtest.duplicated(
    ["epi_year", "epi_week"]
).any()

print("✓ Backtest executado.")

display(backtest.head())

✓ Backtest executado.


,epi_year,epi_week,record_count,historical_median,q75,historical_years,has_minimum_history,median_threshold,ratio_to_median,signal_status
0,2024,1,285,887.0,3696.0,5,True,1330.5,0.321308,NO_SIGNAL
1,2024,2,276,667.0,4407.0,5,True,1000.5,0.413793,NO_SIGNAL
2,2024,3,330,524.0,4053.0,5,True,786.0,0.629771,NO_SIGNAL
3,2024,4,274,452.0,3779.0,5,True,678.0,0.606195,NO_SIGNAL
4,2024,5,363,504.0,3317.0,5,True,756.0,0.720238,NO_SIGNAL


In [30]:
signal_summary = (
    backtest
    .groupby(
        ["epi_year", "signal_status"]
    )
    .size()
    .unstack(fill_value=0)
)

display(signal_summary)

signal_status,NO_SIGNAL,SIGNAL
epi_year,,
2024,52,0
2025,46,6


In [31]:
historical_signals = (
    backtest.loc[
        backtest["signal_status"] == "SIGNAL",
        [
            "epi_year",
            "epi_week",
            "record_count",
            "historical_median",
            "q75",
            "median_threshold",
            "ratio_to_median",
            "historical_years",
        ],
    ]
    .copy()
)

display(historical_signals)

,epi_year,epi_week,record_count,historical_median,q75,median_threshold,ratio_to_median,historical_years
68,2025,17,1087,695.0,1063.50,1042.50,1.564029,6
69,2025,18,1586,744.5,1178.00,1116.75,2.130289,6
70,2025,19,1941,847.0,1285.25,1270.50,2.291617,6
71,2025,20,2201,986.0,1487.75,1479.00,2.232252,6
72,2025,21,2404,1029.5,1558.00,1544.25,2.335114,6
73,2025,22,2148,1145.0,1777.75,1717.50,1.875983,6


In [32]:
assert (
    historical_signals["record_count"]
    > historical_signals["q75"]
).all()

assert (
    historical_signals["record_count"]
    >= historical_signals["median_threshold"]
).all()

print("✓ Todos os sinais históricos satisfazem a regra V1.")

✓ Todos os sinais históricos satisfazem a regra V1.


In [33]:
assert backtest.loc[
    backtest["epi_year"] == 2024,
    "historical_years",
].eq(5).all()

assert backtest.loc[
    backtest["epi_year"] == 2025,
    "historical_years",
].eq(6).all()

print("✓ Backtest sem uso de anos futuros.")

✓ Backtest sem uso de anos futuros.


### Evidências externas — evento de referência 2025

O evento de referência de 2025 foi definido a partir de publicações
independentes da regra de sinais implementada pelo Sentinela.

As publicações foram utilizadas para identificar períodos em que fontes
externas documentaram crescimento ou atividade relevante de SRAG em
Minas Gerais ou Belo Horizonte.

#### Evidências

**11/04/2025 — InfoGripe: alerta para aumento de SRAG**

A publicação referente à SE 14 indicava Belo Horizonte entre as capitais
com sinal de crescimento de SRAG na tendência de longo prazo.

Fonte:
[InfoGripe — crescimento de SRAG em abril de 2025](https://agenciagov.ebc.com.br/noticias/202504/infogripe-alerta-para-aumento-de-srag-em-quase-todas-as-regioes-do-pais)

---

**Maio/2025 — InfoGripe: Influenza A segue em crescimento**

Publicação posterior manteve Belo Horizonte entre os locais com atividade
relevante/crescimento de SRAG, fornecendo evidência de continuidade do
período observado.

Fonte:
[InfoGripe — situação até a SE 20 de 2025](https://agenciagov.ebc.com.br/noticias/202505/infogripe-influenza-a-segue-em-crescimento-no-brasil)
---

**Junho/2025 — InfoGripe: situação até a SE 22**

A publicação referente ao período até a SE 22 incluiu Minas Gerais entre
as UFs com incidência em nível de alerta, risco ou alto risco acompanhada
de tendência de crescimento.

Fonte:
[InfoGripe — situação até a SE 22 de 2025](https://agenciagov.ebc.com.br/noticias/202506/infogripe-casos-de-influenza-a-e-vsr-seguem-em-crescimento-no-pais)

#### Uso no backtest

Essas evidências são externas e independentes da regra V1.

Os sinais produzidos pelo Sentinela não foram utilizados para escolher
as semanas do evento de referência.

A presença de um evento de referência indica um período documentado de
crescimento ou atividade relevante de SRAG e não deve ser interpretada
como confirmação de surto epidemiológico.

In [38]:
reference_events = pd.DataFrame(
    [
        {
            "event_id": "MG_2024_01",
            "region": "MG",
            "epi_year": 2024,
            "start_week": 15,
            "end_week": pd.NA,
            "start_definition": "first_external_documented_growth",
            "reference_source": "InfoGripe",
            "reference_description": (
                "Minas Gerais com sinal de crescimento de SRAG "
                "na tendência de longo prazo."
            ),
        },
        {
            "event_id": "MG_2025_01",
            "region": "MG",
            "epi_year": 2025,
            "start_week": 14,
            "end_week": pd.NA,
            "start_definition": "first_external_documented_growth",
            "reference_source": "InfoGripe",
            "reference_description": (
                "Minas Gerais com incidência em alerta, risco "
                "ou alto risco e crescimento de longo prazo."
            ),
        },
    ]
)

reference_events["start_week"] = (
    reference_events["start_week"].astype("Int64")
)

reference_events["end_week"] = (
    reference_events["end_week"].astype("Int64")
)

display(reference_events)

,event_id,region,epi_year,start_week,end_week,start_definition,reference_source,reference_description
0,MG_2024_01,MG,2024,15,<NA>,first_external_documented_growth,InfoGripe,Minas Gerais com sinal de crescimento de SRAG ...
1,MG_2025_01,MG,2025,14,<NA>,first_external_documented_growth,InfoGripe,"Minas Gerais com incidência em alerta, risco o..."


In [39]:
reference_evidence = pd.DataFrame(
    [
        {
            "event_id": "MG_2024_01",
            "epi_week": 13,
            "evidence_role": "pre_event",
            "geographic_scope": "MG",
            "finding": (
                "MG não listado entre as UFs com crescimento "
                "de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
        },
        {
            "event_id": "MG_2024_01",
            "epi_week": 15,
            "evidence_role": "event_start",
            "geographic_scope": "MG",
            "finding": (
                "MG listado entre as UFs com crescimento "
                "de SRAG de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
        },
        {
            "event_id": "MG_2025_01",
            "epi_week": 14,
            "evidence_role": "event_start",
            "geographic_scope": "MG",
            "finding": (
                "MG em alerta/risco/alto risco com crescimento "
                "de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
        },
        {
            "event_id": "MG_2025_01",
            "epi_week": 25,
            "evidence_role": "event_continuation",
            "geographic_scope": "MG",
            "finding": (
                "MG ainda em alerta/risco/alto risco com "
                "crescimento de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
        },
    ]
)

display(reference_evidence)

,event_id,epi_week,evidence_role,geographic_scope,finding,source
0,MG_2024_01,13,pre_event,MG,MG não listado entre as UFs com crescimento de...,InfoGripe / Agência Gov
1,MG_2024_01,15,event_start,MG,MG listado entre as UFs com crescimento de SRA...,InfoGripe / Agência Gov
2,MG_2025_01,14,event_start,MG,MG em alerta/risco/alto risco com crescimento ...,InfoGripe / Agência Gov
3,MG_2025_01,25,event_continuation,MG,MG ainda em alerta/risco/alto risco com cresci...,InfoGripe / Agência Gov


## Regra de avaliação dos eventos

Para cada evento de referência:

- considera-se `DETECTED` quando existe pelo menos um `SIGNAL` no mesmo
  ano epidemiológico a partir da semana de início documentada;
- o primeiro sinal posterior ou coincidente com o início é utilizado para
  calcular o tempo de detecção;
- `detection_delay_weeks = first_signal_week - start_week`;
- valor igual a zero representa detecção na mesma semana;
- valor positivo representa detecção após o início documentado;
- a ausência de sinal resulta em `NOT_DETECTED`.

Como o encerramento dos eventos de referência não está definido, sinais
fora dos eventos não serão classificados automaticamente como falsos
alertas nesta etapa.

Essa abordagem evita inferir o término de um evento a partir da própria
regra que está sendo avaliada.

In [44]:
event_evaluations = []

for _, event in reference_events.iterrows():
    year = int(event["epi_year"])
    start_week = int(event["start_week"])

    year_signals = backtest.loc[
        (backtest["epi_year"] == year)
        & (backtest["signal_status"] == "SIGNAL")
        & (backtest["epi_week"] >= start_week)
    ].copy()

    if year_signals.empty:
        first_signal_week = pd.NA
        detection_delay_weeks = pd.NA
        detection_status = "NOT_DETECTED"
    else:
        first_signal_week = int(
            year_signals["epi_week"].min()
        )

        detection_delay_weeks = (
            first_signal_week - start_week
        )

        detection_status = "DETECTED"

    event_evaluations.append(
        {
            "event_id": event["event_id"],
            "region": event["region"],
            "epi_year": year,
            "event_start_week": start_week,
            "first_signal_week": first_signal_week,
            "detection_delay_weeks": detection_delay_weeks,
            "detection_status": detection_status,
        }
    )

event_evaluation = pd.DataFrame(event_evaluations)

event_evaluation["first_signal_week"] = (
    event_evaluation["first_signal_week"].astype("Int64")
)

event_evaluation["detection_delay_weeks"] = (
    event_evaluation["detection_delay_weeks"].astype("Int64")
)

display(event_evaluation)

,event_id,region,epi_year,event_start_week,first_signal_week,detection_delay_weeks,detection_status
0,MG_2024_01,MG,2024,15,<NA>,<NA>,NOT_DETECTED
1,MG_2025_01,MG,2025,14,17,3,DETECTED


In [45]:
total_events = len(event_evaluations)

detected_events = (
    event_evaluation["detection_status"]
    .eq("DETECTED")
    .sum()
)

not_detected_events = (
    event_evaluation["detection_status"]
    .eq("NOT_DETECTED")
    .sum()
)

detection_rate = (
    detected_events / total_events
    if total_events > 0
    else float("nan")
)

print(f"Eventos de referência: {total_events}")
print(f"Eventos detectados: {detected_events}")
print(f"Eventos não detectados: {not_detected_events}")
print(f"Proporção detectada: {detection_rate:.1%}")

Eventos de referência: 2
Eventos detectados: 1
Eventos não detectados: 1
Proporção detectada: 50.0%


### Avaliação de falsos alertas

A avaliação de falsos alertas foi realizada somente em semanas para as
quais havia classificação externa explícita e independente da regra V1.

Foram avaliados 8 pontos externos:

- 5 classificados como `EVENT`;
- 3 classificados como `NO_EVENT`.

Nos 3 pontos negativos avaliáveis, a regra V1 não produziu sinais,
resultando em:

- 3 verdadeiros negativos;
- 0 falsos positivos;
- taxa observada de falsos positivos de 0%.

Esse resultado se aplica exclusivamente aos pontos externos avaliados.
O número de pontos negativos é pequeno e semanas sem evidência externa
suficiente não foram consideradas negativas.

Portanto, o resultado não deve ser interpretado como evidência de uma
taxa geral de falsos alertas de 0% ou de especificidade geral de 100%.

Na avaliação dos pontos positivos, houve 1 verdadeiro positivo e
4 falsos negativos entre 5 pontos classificados externamente como
`EVENT`.

As classificações representam concordância com a referência externa
adotada e não confirmação ou exclusão de surtos epidemiológicos.

In [46]:
assert len(event_evaluation) == len(reference_events)

assert set(
    event_evaluation["detection_status"]
) <= {"DETECTED", "NOT_DETECTED"}

detected = event_evaluation.loc[
    event_evaluation["detection_status"] == "DETECTED"
]

not_detected = event_evaluation.loc[
    event_evaluation["detection_status"] == "NOT_DETECTED"
]

assert detected["first_signal_week"].notna().all()
assert detected["detection_delay_weeks"].notna().all()

assert not_detected["first_signal_week"].isna().all()
assert not_detected["detection_delay_weeks"].isna().all()

assert (
    detected["detection_delay_weeks"]
    == (
        detected["first_signal_week"]
        - detected["event_start_week"]
    )
).all()

print("✓ Avaliação dos eventos validada.")
print(f"Eventos avaliados: {len(event_evaluation)}")
print(f"Detectados: {len(detected)}")
print(f"Não detectados: {len(not_detected)}")



✓ Avaliação dos eventos validada.
Eventos avaliados: 2
Detectados: 1
Não detectados: 1


In [47]:
evaluation_summary = pd.DataFrame(
    [
        {
            "reference_events": len(event_evaluation),
            "detected_events": len(detected),
            "not_detected_events": len(not_detected),
            "detection_proportion": (
                len(detected) / len(event_evaluation)
            ),
            "median_detection_delay_weeks": (
                detected["detection_delay_weeks"].median()
                if not detected.empty
                else pd.NA
            ),
            "false_alert_rate": pd.NA,
        }
    ]
)

display(evaluation_summary)

,reference_events,detected_events,not_detected_events,detection_proportion,median_detection_delay_weeks,false_alert_rate
0,2,1,1,0.5,3.0,<NA>


In [48]:
output_dir = project_root / "data" / "processed"

events_path = (
    output_dir
    / "srag_mg_reference_events_2024_2025.csv"
)

event_evaluation_path = (
    output_dir
    / "srag_mg_historical_alert_evaluation.csv"
)

evaluation_summary_path = (
    output_dir
    / "srag_mg_historical_alert_evaluation_summary.csv"
)

reference_events.to_csv(
    events_path,
    sep=";",
    index=False,
    encoding="utf-8",
    na_rep="",
)

event_evaluation.to_csv(
    event_evaluation_path,
    sep=";",
    index=False,
    encoding="utf-8",
    na_rep="",
)

evaluation_summary.to_csv(
    evaluation_summary_path,
    sep=";",
    index=False,
    encoding="utf-8",
    na_rep="",
)

print("✓ Resultados exportados.")
print(events_path)
print(event_evaluation_path)
print(evaluation_summary_path)

✓ Resultados exportados.
/home/matheuspessoa/Sentinela/data/processed/srag_mg_reference_events_2024_2025.csv
/home/matheuspessoa/Sentinela/data/processed/srag_mg_historical_alert_evaluation.csv
/home/matheuspessoa/Sentinela/data/processed/srag_mg_historical_alert_evaluation_summary.csv


## Resultado da avaliação histórica

A regra V1 foi avaliada retrospectivamente nos anos de 2024 e 2025,
utilizando somente anos anteriores para construir a referência de cada
ano avaliado.

Foram definidos dois eventos externos de referência:

- `MG_2024_01`: início documentado na SE 15 de 2024;
- `MG_2025_01`: início documentado na SE 14 de 2025.

### Resultados

Em 2024, a regra V1 não produziu sinais durante o ano. O evento externo
de referência iniciado na SE 15 foi, portanto, classificado como
`NOT_DETECTED`.

Em 2025, a regra produziu seu primeiro sinal na SE 17. Como o evento
externo de referência possui início documentado na SE 14, o evento foi
classificado como `DETECTED`, com atraso de 3 semanas em relação ao
início documentado.

Dos dois eventos externos avaliados, um foi detectado e um não foi
detectado, correspondendo a uma proporção observada de detecção de 50%.

Essa proporção é descritiva deste conjunto de avaliação, composto por
apenas dois eventos, e não deve ser interpretada como uma estimativa
robusta de sensibilidade da regra.

### Falsos alertas

A taxa de falsos alertas não foi estimada.

Embora os eventos possuam início documentado por fontes externas, seus
encerramentos não foram estabelecidos de forma independente. Dessa forma,
não existem janelas negativas suficientemente definidas para classificar
com segurança um sinal como falso alerta.

### Antecedência e atraso

Nenhum evento avaliado apresentou detecção anterior ao início externo
documentado.

O evento de 2025 apresentou primeiro sinal 3 semanas após o início de
referência, caracterizando atraso de detecção, e não antecedência.

### Limitações

A avaliação contém somente dois eventos externos de referência e dois
anos de avaliação.

Os eventos externos representam períodos documentados de crescimento ou
atividade relevante de SRAG e não constituem necessariamente confirmação
de surto epidemiológico.

A ausência de um sinal não demonstra ausência de atividade epidemiológica,
assim como a presença de um `SIGNAL` não confirma um surto.

A regra V1 não deve ser reajustada utilizando os resultados de 2024 e
2025 sem uma nova separação entre desenvolvimento e avaliação, pois isso
introduziria informação do conjunto de avaliação no desenvolvimento do
método.

## Classificação externa das semanas

A avaliação de falsos alertas utiliza somente semanas para as quais existe
evidência externa suficientemente clara.

Estados possíveis:

- `EVENT`: evidência externa explícita de crescimento de SRAG em Minas Gerais;
- `NO_EVENT`: evidência externa explícita de ausência de crescimento, queda
  ou estabilidade segundo o critério adotado;
- `UNKNOWN`: evidência insuficiente ou ambígua.

Semanas `UNKNOWN` são excluídas do cálculo de falsos alertas.

A ausência de publicação ou de menção a Minas Gerais não é interpretada
como `NO_EVENT`.

In [ ]:
reference_timeline = pd.DataFrame(
    [
        {
            "epi_year": 2024,
            "epi_week": 13,
            "external_status": "NO_EVENT",
            "finding": (
                "MG não estava entre as UFs com crescimento de SRAG "
                "na tendência de longo prazo; cenário nacional em queda."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202404/"
                "infogripe-aponta-queda-de-srag-e-aumento-de-casos-"
                "de-vsr-e-gripe"
            ),
        },
        {
            "epi_year": 2024,
            "epi_week": 15,
            "external_status": "EVENT",
            "finding": (
                "MG listado entre as UFs com crescimento de SRAG "
                "na tendência de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202404/"
                "infogripe-internacoes-por-vsr-e-influenza-a-"
                "seguem-em-alta"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 14,
            "external_status": "EVENT",
            "finding": (
                "MG em alerta/risco/alto risco com crescimento "
                "na tendência de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202504/"
                "infogripe-alerta-para-aumento-de-srag-em-quase-"
                "todas-as-regioes-do-pais"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 20,
            "external_status": "EVENT",
            "finding": (
                "Continuidade externa documentada do período "
                "de crescimento de SRAG."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/login?came_from=/noticias/202505/
                "infogripe-influenza-a-segue-em-crescimento-no-brasil"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 24,
            "external_status": "EVENT",
            "finding": (
                "MG ainda listado entre as UFs com crescimento "
                "na tendência de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202506/"
                "infogripe-casos-de-srag-seguem-em-crescimento-"
                "em-parte-do-pais"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 31,
            "external_status": "NO_EVENT",
            "finding": (
                "MG com incidência elevada, porém sem sinal de "
                "crescimento na tendência de longo prazo."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202508/"
                "infogripe-casos-de-vsr-em-criancas-se-mantem-"
                "em-niveis-elevados"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 33,
            "external_status": "NO_EVENT",
            "finding": (
                "Nenhuma UF apresentava crescimento na tendência "
                "de longo prazo; MG permanecia elevado, mas sem crescimento."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202508/"
                "infogripe-aponta-manutencao-de-queda-do-numero-"
                "de-casos-de-srag-em-quase-todo-o-pais"
            ),
        },
        {
            "epi_year": 2025,
            "epi_week": 26,
            "external_status": "EVENT",
            "finding": (
                "MG ainda apresentava crescimento de SRAG associado "
                "à influenza A em jovens, adultos e idosos, apesar de "
                "queda/interrupção do VSR em crianças pequenas."
            ),
            "source": "InfoGripe / Agência Gov",
            "url": (
                "https://agenciagov.ebc.com.br/noticias/202507/"
                "boletim-infogripe-mostra-que-a-influenza-a-e-"
                "responsavel-por-74-dos-obitos-por-srag"
            ),
        }
    ]
)

display(reference_timeline)

,epi_year,epi_week,external_status,finding,source,url
0,2024,13,NO_EVENT,MG não estava entre as UFs com crescimento de ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202404/...
1,2024,15,EVENT,MG listado entre as UFs com crescimento de SRA...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202404/...
2,2025,14,EVENT,MG em alerta/risco/alto risco com crescimento ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202504/...
3,2025,20,EVENT,Continuidade externa documentada do período de...,InfoGripe / Agência Gov,NaN
4,2025,24,EVENT,MG ainda listado entre as UFs com crescimento ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202506/...
5,2025,31,NO_EVENT,"MG com incidência elevada, porém sem sinal de ...",InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202508/...
6,2025,33,NO_EVENT,Nenhuma UF apresentava crescimento na tendênci...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202508/...
7,2025,26,EVENT,MG ainda apresentava crescimento de SRAG assoc...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202507/...


In [54]:
reference_timeline.loc[
    (reference_timeline["epi_year"] == 2025)
    & (reference_timeline["epi_week"] == 20),
    "url",
] = (
    "https://agenciagov.ebc.com.br/noticias/202505/"
    "infogripe-influenza-a-segue-em-crescimento-no-brasil"
)

reference_timeline = (
    reference_timeline
    .sort_values(["epi_year", "epi_week"])
    .reset_index(drop=True)
)

assert reference_timeline["url"].notna().all()

display(reference_timeline)

,epi_year,epi_week,external_status,finding,source,url
0,2024,13,NO_EVENT,MG não estava entre as UFs com crescimento de ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202404/...
1,2024,15,EVENT,MG listado entre as UFs com crescimento de SRA...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202404/...
2,2025,14,EVENT,MG em alerta/risco/alto risco com crescimento ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202504/...
3,2025,20,EVENT,Continuidade externa documentada do período de...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202505/...
4,2025,24,EVENT,MG ainda listado entre as UFs com crescimento ...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202506/...
5,2025,26,EVENT,MG ainda apresentava crescimento de SRAG assoc...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202507/...
6,2025,31,NO_EVENT,"MG com incidência elevada, porém sem sinal de ...",InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202508/...
7,2025,33,NO_EVENT,Nenhuma UF apresentava crescimento na tendênci...,InfoGripe / Agência Gov,https://agenciagov.ebc.com.br/noticias/202508/...


In [57]:
external_evaluation = reference_timeline.merge(
    backtest[
        [
            "epi_year",
            "epi_week",
            "record_count",
            "signal_status",
        ]
    ],
    on=["epi_year", "epi_week"],
    how="left",
    validate="one_to_one",
)

assert external_evaluation["signal_status"].notna().all()

display(
    external_evaluation[
        [
            "epi_year",
            "epi_week",
            "external_status",
            "record_count",
            "signal_status",
        ]
    ]
)

,epi_year,epi_week,external_status,record_count,signal_status
0,2024,13,NO_EVENT,538,NO_SIGNAL
1,2024,15,EVENT,662,NO_SIGNAL
2,2025,14,EVENT,774,NO_SIGNAL
3,2025,20,EVENT,2201,SIGNAL
4,2025,24,EVENT,1812,NO_SIGNAL
5,2025,26,EVENT,1358,NO_SIGNAL
6,2025,31,NO_EVENT,717,NO_SIGNAL
7,2025,33,NO_EVENT,656,NO_SIGNAL


### Classificação dos pontos externos

As classificações `TRUE_POSITIVE`, `FALSE_POSITIVE`,
`TRUE_NEGATIVE` e `FALSE_NEGATIVE` representam concordância ou
discordância entre a regra V1 e os pontos de referência externos
selecionados.

Elas não representam confirmação ou exclusão de surtos.

Somente semanas com classificação externa explícita `EVENT` ou
`NO_EVENT` são avaliadas. Semanas sem evidência externa suficiente
permanecem fora desta análise.

In [59]:
def classify_evaluation(row):
    external = row["external_status"]
    signal = row["signal_status"]

    if external == "EVENT" and signal == "SIGNAL":
        return "TRUE_POSITIVE"

    if external == "EVENT" and signal == "NO_SIGNAL":
        return "FALSE_NEGATIVE"

    if external == "NO_EVENT" and signal == "SIGNAL":
        return "FALSE_POSITIVE"

    if external == "NO_EVENT" and signal == "NO_SIGNAL":
        return "TRUE_NEGATIVE"

    return "NOT_EVALUABLE"


external_evaluation["evaluation_result"] = (
    external_evaluation.apply(
        classify_evaluation,
        axis=1,
    )
)

display(
    external_evaluation[
        [
            "epi_year",
            "epi_week",
            "external_status",
            "signal_status",
            "evaluation_result",
        ]
    ]
)

,epi_year,epi_week,external_status,signal_status,evaluation_result
0,2024,13,NO_EVENT,NO_SIGNAL,TRUE_NEGATIVE
1,2024,15,EVENT,NO_SIGNAL,FALSE_NEGATIVE
2,2025,14,EVENT,NO_SIGNAL,FALSE_NEGATIVE
3,2025,20,EVENT,SIGNAL,TRUE_POSITIVE
4,2025,24,EVENT,NO_SIGNAL,FALSE_NEGATIVE
5,2025,26,EVENT,NO_SIGNAL,FALSE_NEGATIVE
6,2025,31,NO_EVENT,NO_SIGNAL,TRUE_NEGATIVE
7,2025,33,NO_EVENT,NO_SIGNAL,TRUE_NEGATIVE


In [60]:
evaluation_counts = (
    external_evaluation["evaluation_result"]
    .value_counts()
    .reindex(
        [
            "TRUE_POSITIVE",
            "FALSE_POSITIVE",
            "TRUE_NEGATIVE",
            "FALSE_NEGATIVE",
        ],
        fill_value=0,
    )
)

tp = int(evaluation_counts["TRUE_POSITIVE"])
fp = int(evaluation_counts["FALSE_POSITIVE"])
tn = int(evaluation_counts["TRUE_NEGATIVE"])
fn = int(evaluation_counts["FALSE_NEGATIVE"])

positive_points = tp + fn
negative_points = tn + fp

sensitivity = (
    tp / positive_points
    if positive_points > 0
    else float("nan")
)

false_positive_rate = (
    fp / negative_points
    if negative_points > 0
    else float("nan")
)

specificity = (
    tn / negative_points
    if negative_points > 0
    else float("nan")
)

evaluation_metrics = pd.DataFrame(
    [
        {
            "evaluated_points": tp + fp + tn + fn,
            "positive_reference_points": positive_points,
            "negative_reference_points": negative_points,
            "true_positive": tp,
            "false_positive": fp,
            "true_negative": tn,
            "false_negative": fn,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "false_positive_rate": false_positive_rate,
        }
    ]
)

display(evaluation_metrics)

,evaluated_points,positive_reference_points,negative_reference_points,true_positive,false_positive,true_negative,false_negative,sensitivity,specificity,false_positive_rate
0,8,5,3,1,0,3,4,0.2,1.0,0.0


In [61]:
assert tp == 1
assert fp == 0
assert tn == 3
assert fn == 4

assert tp + fn == 5
assert tn + fp == 3

assert len(external_evaluation) == 8

assert external_evaluation[
    "evaluation_result"
].isin(
    [
        "TRUE_POSITIVE",
        "FALSE_POSITIVE",
        "TRUE_NEGATIVE",
        "FALSE_NEGATIVE",
    ]
).all()

print("✓ Avaliação externa validada.")
print(f"Pontos avaliados: {len(external_evaluation)}")
print(f"TP: {tp}")
print(f"FP: {fp}")
print(f"TN: {tn}")
print(f"FN: {fn}")
print(f"Sensibilidade observada: {sensitivity:.1%}")
print(f"Especificidade observada: {specificity:.1%}")
print(
    "Taxa observada de falsos alertas: "
    f"{false_positive_rate:.1%}"
)

✓ Avaliação externa validada.
Pontos avaliados: 8
TP: 1
FP: 0
TN: 3
FN: 4
Sensibilidade observada: 20.0%
Especificidade observada: 100.0%
Taxa observada de falsos alertas: 0.0%


In [63]:
output_dir = project_root / "data" / "processed"

exports = {
    "reference_events": (
        reference_events,
        output_dir / "srag_mg_reference_events_2024_2025.csv",
    ),
    "reference_timeline": (
        reference_timeline,
        output_dir / "srag_mg_reference_timeline_2024_2025.csv",
    ),
    "event_evaluation": (
        event_evaluation,
        output_dir / "srag_mg_event_evaluation_2024_2025.csv",
    ),
    "external_evaluation": (
        external_evaluation,
        output_dir / "srag_mg_external_evaluation_2024_2025.csv",
    ),
    "evaluation_metrics": (
        evaluation_metrics,
        output_dir / "srag_mg_alert_evaluation_metrics.csv",
    ),
}

for name, (df, path) in exports.items():
    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
        na_rep="",
    )

    print(f"✓ {name}: {path}")

✓ reference_events: /home/matheuspessoa/Sentinela/data/processed/srag_mg_reference_events_2024_2025.csv
✓ reference_timeline: /home/matheuspessoa/Sentinela/data/processed/srag_mg_reference_timeline_2024_2025.csv
✓ event_evaluation: /home/matheuspessoa/Sentinela/data/processed/srag_mg_event_evaluation_2024_2025.csv
✓ external_evaluation: /home/matheuspessoa/Sentinela/data/processed/srag_mg_external_evaluation_2024_2025.csv
✓ evaluation_metrics: /home/matheuspessoa/Sentinela/data/processed/srag_mg_alert_evaluation_metrics.csv


In [64]:
assert len(reference_events) == 2
assert len(event_evaluation) == 2

assert (
    event_evaluation["detection_status"]
    .value_counts()
    .to_dict()
    == {
        "NOT_DETECTED": 1,
        "DETECTED": 1,
    }
)

# Pontos externos
assert len(external_evaluation) == 8

expected_counts = {
    "TRUE_POSITIVE": 1,
    "FALSE_POSITIVE": 0,
    "TRUE_NEGATIVE": 3,
    "FALSE_NEGATIVE": 4,
}

actual_counts = (
    external_evaluation["evaluation_result"]
    .value_counts()
    .reindex(expected_counts, fill_value=0)
    .to_dict()
)

assert actual_counts == expected_counts

# 2025
event_2025 = event_evaluation.loc[
    event_evaluation["event_id"] == "MG_2025_01"
].iloc[0]

assert event_2025["first_signal_week"] == 17
assert event_2025["detection_delay_weeks"] == 3

print("=" * 55)
print("RESUMO DA AVALIAÇÃO HISTÓRICA")
print("=" * 55)

print("\nAvaliação por evento:")
print("Eventos externos: 2")
print("Detectados: 1")
print("Não detectados: 1")
print("2025: primeiro sinal 3 semanas após o início documentado")

print("\nAvaliação por pontos externos:")
print("Pontos avaliados: 8")
print("TP: 1")
print("FP: 0")
print("TN: 3")
print("FN: 4")

print("\nLimitações:")
print("- Apenas dois eventos independentes avaliados")
print("- Apenas três pontos externos negativos")
print("- Semanas sem evidência externa foram excluídas")
print("- SIGNAL não representa confirmação de surto")

print("\n✓ Avaliação histórica reproduzida e validada.")
print("=" * 55)

RESUMO DA AVALIAÇÃO HISTÓRICA

Avaliação por evento:
Eventos externos: 2
Detectados: 1
Não detectados: 1
2025: primeiro sinal 3 semanas após o início documentado

Avaliação por pontos externos:
Pontos avaliados: 8
TP: 1
FP: 0
TN: 3
FN: 4

Limitações:
- Apenas dois eventos independentes avaliados
- Apenas três pontos externos negativos
- Semanas sem evidência externa foram excluídas
- SIGNAL não representa confirmação de surto

✓ Avaliação histórica reproduzida e validada.


## Resultado

A regra V1 foi avaliada retrospectivamente utilizando referências
históricas construídas somente com dados anteriores ao ano avaliado.

### Avaliação por evento

- Eventos externos avaliados: 2
- Detectados: 1
- Não detectados: 1
- Evento de 2025: primeiro sinal 3 semanas após o início documentado

### Avaliação por pontos externos

- Pontos avaliados: 8
- Verdadeiros positivos (TP): 1
- Falsos positivos (FP): 0
- Verdadeiros negativos (TN): 3
- Falsos negativos (FN): 4

### Limitações

- Apenas dois eventos independentes foram avaliados.
- Apenas três pontos externos negativos estavam disponíveis.
- Semanas sem evidência externa suficiente foram excluídas.
- Os resultados por ponto são complementares à avaliação por evento.
- `SIGNAL` representa um sinal estatístico e não confirmação de surto.